In [ ]:
!pip install numpy matplotlib networkx
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
grid_size = 25

def create_building():
    grid = np.ones((grid_size, grid_size))

    corridors = [
        (slice(1,-1), 1),
        (1, slice(1,-1)),
        (slice(1,-1), 6),
        (6, slice(6,-1)),
        (slice(6,-1), 12),
        (12, slice(1,12)),
        (slice(12,-1), 18),
        (18, slice(6,18)),
        (18, slice(18,24)),
        (slice(18,24), 23)
    ]

    for r, c in corridors:
        grid[r, c] = 0

    exits = [(0,1), (24,23)]
    for e in exits:
        grid[e] = 2

    return grid, exits

grid, exits = create_building()

In [ ]:
crowd_density = 0.45

def spawn_agents():
    walkable = np.argwhere(grid == 0)
    num_agents = int(len(walkable) * crowd_density)
    chosen = walkable[np.random.choice(len(walkable), num_agents, replace=False)]
    return np.array(chosen)

agents = spawn_agents()
print("Total people spawned:", len(agents))

Total people spawned: 63


In [ ]:
def build_graph(grid):
    G = nx.Graph()
    rows, cols = grid.shape

    for x in range(rows):
        for y in range(cols):
            if grid[x, y] != 1:
                G.add_node((x, y))
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx_, ny_ = x + dx, y + dy
                    if 0 <= nx_ < rows and 0 <= ny_ < cols:
                        if grid[nx_, ny_] != 1:
                            G.add_edge((x, y), (nx_, ny_))
    return G

G = build_graph(grid)

def nearest_exit(pos):
    best_path = None
    best_len = float("inf")

    for ex in exits:
        try:
            path = nx.astar_path(G, tuple(pos), ex)
            if len(path) < best_len:
                best_len = len(path)
                best_path = path
        except:
            pass

    return best_path

In [ ]:
def assign_exit(agent_index, agent_pos):
    exit_index = agent_index % len(exits)
    target_exit = exits[exit_index]

    try:
        return nx.astar_path(G, tuple(agent_pos), target_exit)
    except:
        return nearest_exit(agent_pos)

In [ ]:
paths = []

for i, agent in enumerate(agents):
    path = assign_exit(i, agent)
    if path is None:
        path = nearest_exit(agent)
    if path is not None:
        paths.append(path)

if len(paths) == 0:
    raise ValueError("No valid paths found")

steps = max(len(p) for p in paths)

def compute_metrics(frame):
    evacuated = sum(1 for p in paths if frame >= len(p)-1)
    return evacuated

print("Agents with paths:", len(paths))
print("Total frames:", steps)

Agents with paths: 63
Total frames: 58


In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
plt.close('all')

def update(frame):
    ax.clear()

    ax.imshow(grid, cmap="gray_r")

    current_positions = []

    for path in paths:
        if frame < len(path):
            current_positions.append(path[frame])
        else:
            current_positions.append(path[-1])

    # People (all shown in blue)
    xs = [p[1] for p in current_positions]
    ys = [p[0] for p in current_positions]
    ax.scatter(xs, ys, s=60, color="blue")

    # Exit labels
    exit_names = ["Exit A", "Exit B"]
    for ex, name in zip(exits, exit_names):
        ax.scatter(ex[1], ex[0], s=200, marker="s", color="red")
        ax.text(ex[1], ex[0]-0.6, name,
                color="black", fontsize=10, ha="center")

    # Legend (without evacuated)
    legend_elements = [
        plt.Line2D([0],[0], marker='o', color='blue',
                   label='People', linestyle='', markersize=8),
        plt.Line2D([0],[0], marker='s', color='gray',
                   label='Wall', linestyle='', markersize=10),
        plt.Line2D([0],[0], marker='s', markerfacecolor='white',
                   markeredgecolor='black',
                   label='Walkable Path', linestyle='', markersize=10),
        plt.Line2D([0],[0], marker='s', color='red',
                   label='Exit', linestyle='', markersize=10)
    ]

    ax.legend(handles=legend_elements,
              loc='upper left',
              bbox_to_anchor=(1.02, 1),
              fontsize=9)

    evacuated_count = compute_metrics(frame)

    ax.set_title(f"Step {frame} | Evacuated: {evacuated_count}/{len(paths)}",
                 fontsize=12)

    ax.set_xlim(-0.5, grid_size-0.5)
    ax.set_ylim(grid_size-0.5, -0.5)
    ax.axis("off")

    plt.subplots_adjust(right=0.72)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.pyplot as plt
plt.ioff()
anim = FuncAnimation(fig, update, frames=steps, interval=300)
HTML(anim.to_jshtml())